# GSB 5544 — Topic 4.2: Strings and Regular Expressions  
*Fill each `____` blank as you work; the ✅ checks ask for a sentence or two.*

## The next 20 minutes

| | Question | Where it lands in PA 4.2 |
|---|---|---|
| **a. Strings** | What can plain Python do to one string? | warm-ups 1–5 |
| **b. Patterns** | How do I describe "words like this" instead of one exact word? | warm-up 6, decode 2–7 |
| **c. Series** | How do the same ideas apply to a whole column at once? | everything |

Topic 4.1 crushed text into counts. This topic is the opposite skill: **surgical edits** —
trim this, replace that, keep words matching a pattern. PA 4.2 hands you a scrambled movie
quote; every cleaning step is one of the tools below.

In [ ]:
import pandas as pd
import re

---
## 1. One string: the core methods

A string is a **sequence of characters**, so `len`, indexing, and slicing all work.
Beyond that, six methods cover most real cleaning work:

| Method | What it does |
|---|---|
| `s.lower()` / `s.upper()` | change case |
| `s.strip()` | remove whitespace from **both ends** (not the middle) |
| `s.replace(old, new)` | replace every occurrence, exact text only |
| `s.split(sep)` | string → list of pieces |
| `sep.join(list_of_strings)` | list of pieces → one string (note: called **on the separator**) |
| `s.startswith(x)` / `s.endswith(x)` | True/False tests |

In [ ]:
s = "   The DUDE abides.   "
(
    ____,             # length counts the spaces too
    s.____,          # trim the ends
    s.strip().____,  # methods chain left to right
    s.strip().____
)

In [ ]:
"-".join(["some", "assembly", "required"])     # join is a method of the SEPARATOR

✅ **Check:** strings are *immutable* — `s.strip()` returns a **new** string and leaves `s`
alone. What does `s` contain after running the cell above? What would you write to actually
update it?

**Your answer:** *(write it here — replace this line)*

---
## 2. When exact text isn't enough: regular expressions

`replace("ugh!", "")` removes exactly `ugh!` — but what about `ughhh?` and `ughhhhh,`?
You cannot list every variant. A **regular expression** (regex) describes the whole *family*
of strings with a pattern. The pieces you need this week:

| Pattern | Matches | Example |
|---|---|---|
| `abc` | those literal characters | `ugh` |
| `[aeiou]` | any **one** character from the set | |
| `[^ ...]` | any one character **not** in the set | `[^\w\s]` = not word, not space → **punctuation** |
| `\w` `\s` `.` | word character; whitespace; *anything* | |
| `x+` `x*` `x{2}` | one-or-more; zero-or-more; exactly 2 | `ugh+` = `ugh`, `ughh`, `ughhh`, … |
| `^x` `x$` | at the **start** / at the **end** of the string | `^k` = starts with k; `b$` = ends with b |

The two functions from the `re` module you will use:

- `re.findall(pattern, s)` — list of every non-overlapping match
- `re.sub(pattern, replacement, s)` — replace every match

In [ ]:
groan = "Sighugh! Ughhh, fine. That was rough — very rough, ughhhhh."
re.____(r"ugh+", groan)             # one-or-more h's — catches every groan length

In [ ]:
re.____(r"[Uu]gh+[^\w\s]", "", groan)   # groan + trailing punctuation, gone

Two habits to keep: write patterns as **raw strings** (`r"..."`) so backslashes survive, and
test with `findall` *before* you `sub` — see what you are about to destroy.

✅ **Check:** compare the two outputs to the original sentence. Three surprises to explain:
`findall(r"ugh+")` **missed** `Ughhh,` entirely but matched **inside** `rough` (twice) —
and after the `sub`, the first `rough` survived while the second became `ro`. Why, why, and why?

**Your answer:** *(write it here — replace this line)*

---
## 3. A whole column at once: the `.str` accessor

PA 4.2's `message` is a **pandas Series** of strings, not one string. Two ways to apply
string tools to every element:

1. **`.str` accessor** — most string methods, vectorized: `s.str.strip()`, `s.str.len()`, `s.str.upper()`, `s.str.contains(pat)`, `s.str.replace(pat, repl, regex=True)`, `s.str.slice(0, n)`
2. **`.apply` + `lambda`** — for anything `.str` doesn't offer: `s.apply(lambda w: re.findall(..., w))`

And the crucial difference from the `re` module: in `.str.replace`, **regex is off by default** —
pass `regex=True` when your pattern is a pattern.

In [ ]:
words = pd.Series(["  kite ", "Kayakugh!", "  banana  ", "clarab"])
words.____()

In [ ]:
cleaned = words.str.strip()
(
    cleaned.____(),                                  # length of each word
    cleaned[cleaned.____(r"^k", case=False)],   # starts with k or K
)

In [ ]:
cleaned.str.____(r"b$", "y", ____)           # ends in b -> y : clarab -> claray

✅ **Check:** `words.str.replace("b$", "y")` (no `regex=True`) silently changes **nothing**.
Why no error, and why no change?

**Your answer:** *(write it here — replace this line)*

---
## 4. The PA 4.2 pipeline in miniature

The decode activity is: **strip → surgical regex repairs → truncate → join**. Here is the
whole shape on a three-word message — the PA is this, nine steps instead of three:

In [ ]:
mini = pd.Series(["  Hekko!  ", "ughhh!zhere", "buddb  "])

fixed = (mini
         .str.strip()                              # 1. trim ends
         .str.replace(r"ugh+[^\w\s]", "", regex=True)   # 2. delete the groans
         .str.replace("kk", "ll")                  # 3. literal swap — no regex needed
         .str.replace("z", "t")                    # 4. another literal swap
         .str.replace(r"b$", "y", regex=True))     # 5. pattern swap — regex needed
____                                  # 6. Series -> one string

## The three lines to keep

| | |
|---|---|
| **Strings** | `strip / lower / replace / split`, and `sep.join(pieces)` to reassemble — always reassign the result |
| **Patterns** | `[^\w\s]` punctuation, `+` repeats, `^`/`$` anchors; test with `findall` before you `sub` |
| **Series** | `.str.method()` for the whole column; `regex=True` when the pattern is a pattern; `.apply(lambda ...)` for the rest |

PA 4.2 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).